<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Fine-Tuning and Optimizing a Pretrained Transformer for Text Classification</h1> 


Estimated time needed: **45** minutes


<h2>Objectives</h2><ul><li> Load a pretrained transformer model from Hugging Face, add a classification head, and configure the training loop with Adaptive Moment Estimation with Weight Decay (AdamW) optimizer, learning rate, and number of epochs</li><li> Experiment with at least two learning rate values and implement a learning rate scheduler, observing the effect on training loss convergence</li><li> Evaluate classification performance on a held-out validation set using accuracy and F1-score, and with AI-assisted support, justify the best-performing hyperparameter combination with supporting evidence</li></ul> 


## Table of contents

In this lab, you will fine-tune a pretrained transformer for text classification and compare hyperparameter settings using validation accuracy and F1-score.

<ul>
    <li><a href="#Pretrained-Transformer-Fine-tuning">Pretrained Transformer Fine-tuning</a></li>
    <li><a href="#Create-a-Text-Classification-Dataset">Create a Text Classification Dataset</a></li>
    <li><a href="#Tokenize-the-Dataset">Tokenize the Dataset</a></li>
    <li><a href="#Load-AutoModel-for-Sequence-Classification">Load AutoModel for Sequence Classification</a></li>
    <li><a href="#Configure-AdamW-and-Scheduler">Configure AdamW and Scheduler</a></li>
    <li><a href="#Run-Learning-Rate-Experiments">Run Learning Rate Experiments</a></li>
    <li><a href="#Evaluate-Accuracy-and-F1-score">Evaluate Accuracy and F1-score</a></li>
    <li><a href="#AI-assisted-Hyperparameter-Justification">AI-assisted Hyperparameter Justification</a></li>

</ul>

<hr>


### Installing and importing required libraries


This lab uses Hugging Face Hub models and tokenizers through <code>from_pretrained</code>, so the notebook can be run end to end from your side once network access is available.


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cells__ to install or import them:


In [7]:
# Install PyTorch

!pip install torch

In [2]:
# Install Hugging Face Transformers

!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.5 MB/s eta 0:00:00
Using cached filelock-3.32.4-py3-none-any.whl (99 kB)
Using cached fsspec-2026.7.0-py3-none-any.whl (206 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [transformers]


In [3]:
# Import the libraries we need for this lab

import random

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

random.seed(2)
torch.manual_seed(2)

ModuleNotFoundError: No module named 'torch'

<!--Empty Space for separating topics-->


## Pretrained Transformer Fine-tuning


Fine-tuning adapts a pretrained transformer to a new supervised task. For text classification, the model uses the pretrained encoder and adds a classification head that predicts class labels.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Transformer%2C%20stacked%20layers%20and%20sublayers.png" width="650" alt="Transformer stacked layers and sublayers diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Transformer, stacked layers and sublayers.png by dvgodoy</em></p>


In this lab, you will use <code>AutoModelForSequenceClassification</code>, which loads a pretrained transformer with a classification head already attached for the requested number of labels.


<!--Empty Space for separating topics-->


## Create a Text Classification Dataset


Create a small sentiment dataset. The label <code>1</code> represents positive sentiment, and the label <code>0</code> represents negative sentiment:


In [4]:
# Create a small text classification dataset

positive_texts = [
    "The app is fast and easy to use.",
    "The support team solved my problem quickly.",
    "The product feels reliable and well designed.",
    "The documentation is clear and practical.",
    "The service was friendly and efficient.",
    "The update improved performance and stability.",
    "The interface is clean and intuitive.",
    "The package arrived early and undamaged.",
    "The course examples were helpful and relevant.",
    "The device works exactly as expected.",
]

negative_texts = [
    "The app is slow and difficult to use.",
    "The support team ignored my problem completely.",
    "The product feels fragile and poorly designed.",
    "The documentation is confusing and incomplete.",
    "The service was rude and inefficient.",
    "The update caused crashes and errors.",
    "The interface is cluttered and frustrating.",
    "The package arrived late and damaged.",
    "The course examples were vague and outdated.",
    "The device failed during normal use.",
]

texts = positive_texts + negative_texts
labels = [1] * len(positive_texts) + [0] * len(negative_texts)
examples = list(zip(texts, labels))
random.shuffle(examples)

print("Number of examples:", len(examples))
print(examples[:3])

Number of examples: 20
[('The app is slow and difficult to use.', 0), ('The support team ignored my problem completely.', 0), ('The service was friendly and efficient.', 1)]

Add simple text variations to create more training samples while preserving labels:


In [ ]:
# Expand examples with short prefixes

prefixes = ["", "Overall, ", "In my experience, "]
expanded_examples = []

for text, label in examples:
    for prefix in prefixes:
        expanded_examples.append((prefix + text, label))

random.shuffle(expanded_examples)

print("Expanded examples:", len(expanded_examples))

<!--Empty Space for separating topics-->


## Tokenize the Dataset


Use the tokenizer that matches the pretrained model. This lab uses a compact pretrained model so the fine-tuning experiment can run quickly:


In [ ]:
# Load tokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

Create a custom Dataset that tokenizes each text example and returns tensors for the model:


In [ ]:
# Create a tokenized Dataset

class TransformerTextDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=64):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        text, label = self.examples[index]
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


full_dataset = TransformerTextDataset(expanded_examples, tokenizer)

Split the dataset into training and validation sets, then create DataLoaders:


In [ ]:
# Split dataset and create DataLoaders

train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(2),
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False)

batch = next(iter(train_loader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

<!--Empty Space for separating topics-->


## Load AutoModel for Sequence Classification


<code>AutoModelForSequenceClassification</code> loads the pretrained transformer body and creates a classification head for the number of labels you specify:


In [ ]:
# Load a pretrained transformer with a classification head

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

<!--Empty Space for separating topics-->


## Configure AdamW and Scheduler


AdamW is commonly used for transformer fine-tuning because it combines adaptive moment estimation with decoupled weight decay. A scheduler gradually changes the learning rate during training:


In [ ]:
# Create metric helpers

def compute_accuracy(predictions, labels):
    correct = sum(int(pred == label) for pred, label in zip(predictions, labels))
    return correct / max(len(labels), 1)

def compute_f1_score(predictions, labels):
    tp = sum(1 for pred, label in zip(predictions, labels) if pred == 1 and label == 1)
    fp = sum(1 for pred, label in zip(predictions, labels) if pred == 1 and label == 0)
    fn = sum(1 for pred, label in zip(predictions, labels) if pred == 0 and label == 1)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return 2 * precision * recall / max(precision + recall, 1e-8)

Create training and evaluation functions that use AdamW and a linear learning rate scheduler:


In [ ]:
# Create training and evaluation functions

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            outputs = model(**batch)
            total_loss += outputs.loss.item()
            predictions = outputs.logits.argmax(dim=-1)
            all_predictions.extend(predictions.tolist())
            all_labels.extend(batch["labels"].tolist())

    accuracy = compute_accuracy(all_predictions, all_labels)
    f1_score = compute_f1_score(all_predictions, all_labels)
    return total_loss / len(loader), accuracy, f1_score

<!--Empty Space for separating topics-->


## Run Learning Rate Experiments


Train separate model runs with two learning rates. Each run starts from the same pretrained checkpoint so the comparison is fair:


In [ ]:
# Run a fine-tuning experiment

def run_experiment(learning_rate, epochs=2):
    torch.manual_seed(2)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    total_training_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_training_steps,
    )

    history = []
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        valid_loss, valid_accuracy, valid_f1 = evaluate(model, valid_loader)
        history.append({
            "epoch": epoch + 1,
            "learning_rate": learning_rate,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "valid_accuracy": valid_accuracy,
            "valid_f1": valid_f1,
        })
        print(f"lr={learning_rate} epoch={epoch + 1} train_loss={train_loss:.4f} valid_accuracy={valid_accuracy:.4f} valid_f1={valid_f1:.4f}")
    return model, history

Compare two learning rate values:


In [ ]:
# Compare learning rates

learning_rates = [5e-5, 2e-5]
experiment_histories = []

for learning_rate in learning_rates:
    trained_model, history = run_experiment(learning_rate=learning_rate, epochs=2)
    experiment_histories.extend(history)

<!--Empty Space for separating topics-->


## Evaluate Accuracy and F1-score</h2>


Review validation loss, accuracy, and F1-score for each hyperparameter setting:


In [ ]:
# Display experiment results

for row in experiment_histories:
    print(row)

Select the best-performing hyperparameter combination using F1-score first and validation loss as a tie-breaker:


In [ ]:
# Select the best result

best_result = sorted(
    experiment_histories,
    key=lambda row: (row["valid_f1"], row["valid_accuracy"], -row["valid_loss"]),
    reverse=True,
)[0]

best_result

<!--Empty Space for separating topics-->


## AI-assisted Hyperparameter Justification


AI-assisted support can help summarize experiment evidence, but the recommendation should be tied to validation metrics and convergence behavior:


In [ ]:
# Generate a rule-based hyperparameter justification

def justify_best_result(best_result, histories):
    matching_rows = [row for row in histories if row["learning_rate"] == best_result["learning_rate"]]
    first_loss = matching_rows[0]["train_loss"]
    last_loss = matching_rows[-1]["train_loss"]

    if last_loss < first_loss:
        convergence_note = "training loss decreased across epochs"
    else:
        convergence_note = "training loss did not clearly decrease"

    return (
        f"The best learning rate is {best_result['learning_rate']} because it reached "
        f"validation accuracy {best_result['valid_accuracy']:.4f} and F1-score {best_result['valid_f1']:.4f}. "
        f"This setting is supported by the convergence evidence: {convergence_note}."
    )

print(justify_best_result(best_result, experiment_histories))

When comparing fine-tuning runs, prefer the setting that performs best on the held-out validation set, not the setting that only minimizes training loss.



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--Empty Space for separating topics-->


<h2>About the Author:</h2> 

**Harish Pant** is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations. 


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2026-05-11  | 1.0  | IBM Skills Network  |  Created lab for fine-tuning and optimizing a pretrained transformer for text classification |
-->


<hr>





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
